In [7]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [8]:
df = sns.load_dataset('titanic')

df = df[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']]

display(df.head())
print("\nMissing values per column:\n", df.isnull().sum())

,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S



Missing values per column:
 survived      0
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
dtype: int64


In [9]:
base = df.copy()

base['age'] = base['age'].fillna(base['age'].median())
base['embarked'] = base['embarked'].fillna(base['embarked'].mode()[0])

le = LabelEncoder()
base['sex'] = le.fit_transform(base['sex'])
base['embarked'] = le.fit_transform(base['embarked'])

X = base.drop('survived', axis=1)
y = base['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
base_acc = accuracy_score(y_test, model.predict(X_test))
print(f"Baseline Accuracy: {base_acc:.4f}")

Baseline Accuracy: 0.8027


In [10]:
eng = df.copy()

eng['age'] = eng['age'].fillna(eng['age'].median())
eng['embarked'] = eng['embarked'].fillna(eng['embarked'].mode()[0])

eng['family_size'] = eng['sibsp'] + eng['parch'] + 1
eng['is_alone'] = (eng['family_size'] == 1).astype(int)

eng['age_group'] = pd.cut(eng['age'], bins=[0, 12, 18, 60, 100],
                          labels=['child', 'teen', 'adult', 'senior'])

eng['fare'] = np.log1p(eng['fare'])

eng = pd.get_dummies(eng, columns=['sex', 'embarked', 'age_group'])

num_cols = ['age', 'fare', 'family_size']
eng[num_cols] = StandardScaler().fit_transform(eng[num_cols])

display(eng.head())

,survived,pclass,age,sibsp,parch,fare,family_size,is_alone,sex_female,sex_male,embarked_C,embarked_Q,embarked_S,age_group_child,age_group_teen,age_group_adult,age_group_senior
0,0,3,-0.565736,1,0,-0.879741,0.059160,0,False,True,False,False,True,False,False,True,False
1,1,1,0.663861,1,0,1.361220,0.059160,0,True,False,True,False,False,False,False,True,False
2,1,3,-0.258337,0,0,-0.798540,-0.560975,1,True,False,False,False,True,False,False,True,False
3,1,1,0.433312,1,0,1.062038,0.059160,0,True,False,False,False,True,False,False,True,False
4,0,3,0.433312,0,0,-0.784179,-0.560975,1,False,True,False,False,True,False,False,True,False


In [11]:
X2 = eng.drop('survived', axis=1)
y2 = eng['survived']
X_train, X_test, y_train, y_test = train_test_split(X2, y2, test_size=0.25, random_state=42)

model2 = RandomForestClassifier(random_state=42)
model2.fit(X_train, y_train)
eng_acc = accuracy_score(y_test, model2.predict(X_test))
print(f"Engineered Accuracy: {eng_acc:.4f}")

Engineered Accuracy: 0.7937


In [12]:
print(f"Baseline Accuracy  : {base_acc:.4f}")
print(f"Engineered Accuracy: {eng_acc:.4f}")
print(f"Improvement        : {eng_acc - base_acc:+.4f}")

Baseline Accuracy  : 0.8027
Engineered Accuracy: 0.7937
Improvement        : -0.0090
